# Proyecto Libre – Agente Inversor con LSTM

En este proyecto construiremos un **agente inversor** que sugiere **Comprar / Mantener / Vender** usando un modelo **LSTM**.

> **Aviso**: Este proyecto es educativo y **no es asesoramiento financiero**.

## 1. Carga de datos
Usaremos datos históricos de precios con `yfinance`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import yfinance as yf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

plt.rcParams['figure.figsize'] = (10, 4)


In [ ]:
# Parámetros principales
TICKER = "AAPL"  # puedes cambiarlo por TSLA, BTC-USD, etc.
START = "2016-01-01"
END = "2024-01-01"

# Descarga de datos
df = yf.download(TICKER, start=START, end=END)

df.head()

## 2. Ingeniería de características (features)
Creamos algunas variables sencillas de análisis técnico.

In [ ]:
df = df.copy()

# Retornos
df['Return_1'] = df['Close'].pct_change(1)
df['Return_5'] = df['Close'].pct_change(5)

# Medias móviles
df['MA_10'] = df['Close'].rolling(10).mean()
df['MA_20'] = df['Close'].rolling(20).mean()

# Volatilidad
df['Volatility_10'] = df['Return_1'].rolling(10).std()

# RSI simple
delta = df['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
rs = gain / (loss + 1e-9)
df['RSI_14'] = 100 - (100 / (1 + rs))

# Limpiar nulos
df = df.dropna()
df.head()

## 3. Etiqueta (Buy / Hold / Sell)
Definimos la etiqueta mirando el retorno futuro a `HORIZON` días.

In [ ]:
HORIZON = 5  # días a futuro
THRESHOLD = 0.01  # umbral 1%

# Retorno futuro
df['Future_Return'] = df['Close'].shift(-HORIZON) / df['Close'] - 1

# Etiquetas
# 1 = Buy, 0 = Hold, -1 = Sell
def label_signal(x):
    if x > THRESHOLD:
        return 1
    elif x < -THRESHOLD:
        return -1
    else:
        return 0


df['Signal'] = df['Future_Return'].apply(label_signal)

# Eliminar filas finales sin futuro
df = df.dropna()

df['Signal'].value_counts()

## 4. Preparación para LSTM
Creamos secuencias con una ventana temporal fija.

In [ ]:
FEATURES = ['Close', 'Return_1', 'Return_5', 'MA_10', 'MA_20', 'Volatility_10', 'RSI_14']
WINDOW = 30  # número de días por secuencia

# Escalado
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df[FEATURES])

# Crear secuencias
X, y = [], []
for i in range(WINDOW, len(df)):
    X.append(scaled_features[i-WINDOW:i])
    y.append(df['Signal'].iloc[i])

X = np.array(X)
y = np.array(y)

# Convertir etiquetas a clases 0,1,2
# -1 -> 0, 0 -> 1, 1 -> 2
y = y + 1

print('X shape:', X.shape)
print('y shape:', y.shape)

## 5. División Train/Test
Separamos el dataset de forma temporal (sin mezclar).

In [ ]:
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print('Train:', X_train.shape, 'Test:', X_test.shape)

## 6. Modelo LSTM
Usamos una red LSTM sencilla con regularización.

In [ ]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 7. Entrenamiento
Usamos *EarlyStopping* para evitar sobreajuste.

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

## 8. Evaluación
Mostramos métricas clásicas y matriz de confusión.

In [ ]:
# Predicciones
y_pred = model.predict(X_test)
y_pred_class = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred_class, target_names=['Sell','Hold','Buy']))

cm = confusion_matrix(y_test, y_pred_class)
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Sell','Hold','Buy'], yticklabels=['Sell','Hold','Buy'])
plt.title('Matriz de confusión')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

## 9. Simulación simple de estrategia
Convertimos señales en una estrategia básica para comparar con *Buy & Hold*.

In [ ]:
# Simulación en el tramo de test
prices = df['Close'].iloc[WINDOW+split:WINDOW+len(X)]
returns = prices.pct_change().fillna(0)

# Señales predichas: -1,0,1
signal = y_pred_class - 1

# Estrategia: si Buy -> +1, Sell -> -1, Hold -> 0
strategy_returns = returns * signal

# Cálculo de rendimiento acumulado
capital_strategy = (1 + strategy_returns).cumprod()
capital_buy_hold = (1 + returns).cumprod()

plt.figure(figsize=(8, 4))
plt.plot(capital_strategy.values, label='Estrategia IA')
plt.plot(capital_buy_hold.values, label='Buy & Hold')
plt.title('Comparativa de rendimiento')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 10. Conclusiones
- El LSTM aprende patrones de corto plazo usando precios e indicadores técnicos.
- La evaluación depende mucho del activo y del horizonte.
- La simulación básica sirve como referencia, pero en un proyecto real habría que incluir costes y riesgos.